# SEAS 8515 - Homework 10 (Option 1)
Spark + MLlib

## Student Information
Name:
Date:

## 1. Load Data

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
spark = SparkSession.builder.appName("Homework11_Option1").getOrCreate()

In [3]:
# Load dataset into Spark DataFrame
df = spark.read.csv("energydata_complete.csv", header=True, inferSchema=True)

# Drop date column
df = df.drop("date")

# Show first 10 rows
df.show(10, truncate=False)


+----------+------+----------------+----------------+-----+----------------+-----+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----+----------------+----------------+-----+----------------+----------------+----------------+----------------+----------------+----------------+------------------+------------------+
|Appliances|lights|T1              |RH_1            |T2   |RH_2            |T3   |RH_3            |T4              |RH_4            |T5              |RH_5            |T6              |RH_6            |T7              |RH_7            |T8  |RH_8            |T9              |RH_9 |T_out           |Press_mm_hg     |RH_out          |Windspeed       |Visibility      |Tdewpoint       |rv1               |rv2               |
+----------+------+----------------+----------------+-----+----------------+-----+----------------+----------------+----------------+----------------+--------

## 2. EDA

In [4]:
# Perform summary statistics
# Show correlations
df.describe().show()

target_col = "Appliances"
feature_cols = [c for c in df.columns if c != target_col]

correlations = []
for c in feature_cols:
    correlations.append((c, df.stat.corr(c, target_col)))

spark.createDataFrame(correlations, ["feature", "correlation"]).show()

+-------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+--------------------+--------------------+
|summary|        Appliances|            lights|                T1|              RH_1|                T2|              RH_2|                T3|              RH_3|                T4|             RH_4|                T5|             RH_5|                T6|             RH_6|                T7|             RH_7|                T8|              RH_8|                T9|              RH_9|            T_out|      Press_mm_hg|            RH_out|         Windspeed| 

## 3. Train-Test Split

In [5]:
# Split data 75/25 with seed
trainDF, testDF = df.randomSplit([0.75, 0.25], seed=42)


## 4. Pipeline

In [12]:
# Build pipeline with VectorAssembler and LinearRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

feature_cols = [c for c in df.columns if c != "Appliances"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
lr = LinearRegression(featuresCol="features", labelCol="Appliances")

pipeline = Pipeline(stages=[assembler, lr])
model = pipeline.fit(trainDF)


## 5. Evaluation

In [11]:
# Compute RMSE and R2
# Identify top 5 features
from pyspark.ml.evaluation import RegressionEvaluator

predDF = model.transform(testDF)

r2 = RegressionEvaluator(labelCol="Appliances", predictionCol="prediction", metricName="r2").evaluate(predDF)
rmse = RegressionEvaluator(labelCol="Appliances", predictionCol="prediction", metricName="rmse").evaluate(predDF)
print("R-squared:", r2)
print("RMSE:", rmse)

lrModel = model.stages[-1]
coefficients = lrModel.coefficients.toArray()

feature_importance = list(zip(feature_cols, coefficients))
top5 = sorted(feature_importance, key=lambda x: abs(x[1]), reverse=True)[:5]

print("Top 5 features:")
for f, c in top5:
    print(f, c)


R-squared: 0.15742586683362092
RMSE: 92.14954579658067
Top 5 features:
T3 26.316735730977133
T2 -17.137707591629745
T9 -17.11911889688563
RH_1 14.979481773605663
RH_2 -13.58842008851246


## Explanation

Explain your results and observations.

The R-squared value is 0.157, which means the model is only able to explain about 15.7% of the variation in the Appliances energy consumption. This indicates that the model is not very strong and does not capture the relationship between features and the target very well.

The RMSE value is 92.15, which means that on average, the model’s predictions are off by around 92 units. This is a relatively high error, showing that the predictions are not very accurate.

Overall, the model performance is weak, and it may need better features, more preprocessing, or a different model to improve the results.